# History-based Wine Transaction Anomaly Detection
Supervised chemistry followed by an unsupervised PyTorch autoencoder. Purchase events and scenario labels are synthetic. The notebook loads saved checkpoints; it does not retrain. Scores are reconstruction errors, not fraud probabilities.

In [1]:
import json
import pandas as pd
from history import ROOT, FEATURES
from demo import Screener
events = pd.read_csv(ROOT / "data/events_test.csv")
events.head()

   transaction_id customer_id  timestamp_seconds  ...  anomaly  scenario  event_index
0  test-C0000-T00  test-C0000      303952.799780  ...        0    normal            0
1  test-C0000-T01  test-C0000      315468.194072  ...        0    normal            1
2  test-C0000-T02  test-C0000      359378.381008  ...        0    normal            2
3  test-C0000-T03  test-C0000      366724.718871  ...        0    normal            3
4  test-C0000-T04  test-C0000      384142.488523  ...        0    normal            4

[5 rows x 14 columns]

## Replay a raw customer history
The first five events warm up the state. The current event is scored using prior events only. No anomaly or scenario label is passed to the AE.

In [2]:
def replay(customer, stop_index):
    screen = Screener()
    output = None
    rows = events[(events.customer_id == customer) & (events.event_index <= stop_index)].sort_values("timestamp_seconds")
    for event in rows.to_dict("records"):
        output = screen.predict_event(event)
    return output
replay("test-C0005", 8)

{'transaction_id': 'test-C0005-T08', 'predicted_cultivar': 1, 'declared_cultivar': 1, 'derived_features': {'amount_deviation': -3.3820451634062234, 'quantity_deviation': -2.2908037078902783, 'log_purchases_last_hour': 0.6931471805599453, 'log_minutes_since_previous': 2.4347906997883757, 'new_merchant_category': 1.0, 'log_location_shift_km': 1.0216937376550528, 'log_unit_price': 2.4647325495417056, 'chemical_disagreement': 0.019891560077667236, 'cultivar_mismatch': 0.0}, 'anomaly_score': 5.181547164916992, 'threshold': 4.424379348754883, 'action': 'human_review', 'top_factors': [{'feature': 'new_merchant_category', 'squared_standardized_error': 42.015933990478516, 'share_of_total_error': 0.9009735584259033}, {'feature': 'log_unit_price', 'squared_standardized_error': 3.6365537643432617, 'share_of_total_error': 0.07798086106777191}, {'feature': 'log_location_shift_km', 'squared_standardized_error': 0.495214581489563, 'share_of_total_error': 0.010619191452860832}], 'interpretation': 'Unca

## A different anomaly mechanism
This example has a large location-shift reconstruction error. Explanations rank squared standardized reconstruction errors.

In [3]:
replay("test-C0010", 6)

{'transaction_id': 'test-C0010-T06', 'predicted_cultivar': 0, 'declared_cultivar': 0, 'derived_features': {'amount_deviation': 0.3147277679023193, 'quantity_deviation': 0.2507312727711422, 'log_purchases_last_hour': 0.0, 'log_minutes_since_previous': 7.939306455206929, 'new_merchant_category': 0.0, 'log_location_shift_km': 6.565016046313294, 'log_unit_price': 3.071483235721115, 'chemical_disagreement': 0.0001227855682373047, 'cultivar_mismatch': 0.0}, 'anomaly_score': 5.2837748527526855, 'threshold': 4.424379348754883, 'action': 'human_review', 'top_factors': [{'feature': 'log_location_shift_km', 'squared_standardized_error': 45.69411087036133, 'share_of_total_error': 0.9608893990516663}, {'feature': 'log_minutes_since_previous', 'squared_standardized_error': 1.2614471912384033, 'share_of_total_error': 0.026526641100645065}, {'feature': 'log_unit_price', 'squared_standardized_error': 0.2794784903526306, 'share_of_total_error': 0.00587707944214344}], 'interpretation': 'Uncalibrated reco

## Compare all models on the same held-out data
The supervised comparisons receive training anomaly labels; the AE does not. A better supervised result is reported directly.

In [4]:
metrics = json.loads((ROOT / "results/metrics.json").read_text())
pd.DataFrame([{ "model": name, **{key: value for key, value in result["test"].items()}} for name, result in metrics["results"].items()])

                  model  accuracy  ...  false_positive_rate  flag_rate
0        ae_combined_b3  0.922667  ...             0.029395   0.051733
1   ae_transaction_only  0.917867  ...             0.032277   0.052267
2          mlp_weighted  0.972267  ...             0.003170   0.052800
3        mlp_unweighted  0.969600  ...             0.006916   0.057067
4  mlp_transaction_only  0.972800  ...             0.001729   0.050667
5       linear_combined  0.938400  ...             0.049856   0.105333

[6 rows x 18 columns]

In [5]:
pd.DataFrame({name: result["test_scenario_recall"] for name, result in metrics["results"].items()})

                   ae_combined_b3  ...  linear_combined
amount                   0.571429  ...         0.392857
velocity                 0.022727  ...         0.886364
merchant                 0.933333  ...         0.833333
location                 0.928571  ...         0.785714
chemical_mismatch        0.000000  ...         0.651163
combined                 1.000000  ...         1.000000

[6 rows x 6 columns]

## Threshold and review workload
Thresholds below come from validation-score percentiles. Test consequences are displayed without changing the selected 95th-percentile operating point.

In [6]:
pd.read_csv(ROOT / "results/threshold_tradeoffs.csv")[["model", "validation_percentile", "threshold", "precision", "recall", "fp", "fn"]]

                 model  validation_percentile  threshold  ...    recall   fp   fn
0       ae_combined_b3                   90.0   0.429370  ...  0.578571  418  118
1       ae_combined_b3                   95.0   4.424379  ...  0.328571  102  188
2       ae_combined_b3                   97.5   4.958277  ...  0.225000   35  217
3       ae_combined_b3                   99.0   6.215255  ...  0.103571    0  251
4  ae_transaction_only                   90.0   0.276483  ...  0.610714  234  109
5  ae_transaction_only                   95.0   4.756496  ...  0.300000  112  196
6  ae_transaction_only                   97.5   6.151749  ...  0.185714   42  228
7  ae_transaction_only                   99.0   6.500620  ...  0.117857    8  247

[8 rows x 7 columns]

## Learning and trade-off plots
![AE learning curves](results/ae_combined_b3_learning.png)
![Threshold trade-offs](results/threshold_tradeoffs.png)

The AE trains on 3.16% contaminated transaction data. Its current threshold misses many velocity and chemical-mismatch cases; these failures are part of the measured result. See the report for efficiency, deployment design and complete limits.